<a href="https://colab.research.google.com/github/sumalya41/QFin.Colab/blob/main/financial_regime_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install numpy pandas scipy scikit-learn matplotlib yfinance torch hdbscan hmmlearn lightgbm networkx

In [ ]:
#data engine
%%writefile data_engine.py
"""
data_engine.py

Feature engineering engine for non-stationary financial time-series analysis.

This module constructs a high-dimensional feature stack designed to capture:
- Higher-order distributional moments
- Volatility term structures
- Cross-asset dependency shifts
- Microstructure/liquidity proxies

The architecture intentionally focuses on non-ergodic market behavior where
statistical properties evolve over time.


"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import kurtosis, skew


@dataclass
class DataConfig:
    tickers: List[str]
    start_date: str = "2008-01-01"
    end_date: str | None = None


class MarketDataEngine:
    """
    Data ingestion and feature engineering engine.
    """

    def __init__(self, config: DataConfig):
        self.config = config
        self.raw_data: pd.DataFrame | None = None
        self.features: pd.DataFrame | None = None

    def download_data(self) -> pd.DataFrame:
        """
        Download adjusted OHLCV data from yfinance.
        """

        data = yf.download(
            tickers=self.config.tickers,
            start=self.config.start_date,
            end=self.config.end_date,
            auto_adjust=True,
            progress=False,
        )

        self.raw_data = data
        return data

    @staticmethod
    def compute_log_returns(prices: pd.DataFrame) -> pd.DataFrame:
        return np.log(prices / prices.shift(1))

    @staticmethod
    def rolling_skew(series: pd.Series, window: int) -> pd.Series:
        return series.rolling(window).apply(
            lambda x: skew(x, bias=False), raw=False
        )

    @staticmethod
    def rolling_kurtosis(series: pd.Series, window: int) -> pd.Series:
        return series.rolling(window).apply(
            lambda x: kurtosis(x, fisher=True, bias=False), raw=False
        )

    @staticmethod
    def realized_volatility(returns: pd.Series, window: int) -> pd.Series:
        return returns.rolling(window).std() * np.sqrt(252)

    @staticmethod
    def rolling_autocorr(series: pd.Series, window: int) -> pd.Series:
        return series.rolling(window).apply(
            lambda x: pd.Series(x).autocorr(lag=1),
            raw=False,
        )

    def build_feature_stack(self) -> pd.DataFrame:
        """
        Construct the multi-layered feature matrix.

        All features are shifted by 1 day to avoid leakage.
        """

        if self.raw_data is None:
            raise ValueError("Data not downloaded.")

        close = self.raw_data["Close"]
        volume = self.raw_data["Volume"]

        returns = self.compute_log_returns(close)

        feature_dict: Dict[str, pd.Series] = {}

        # ==========================================================
        # 1. Statistical Moments Layer
        # ==========================================================

        for ticker in close.columns:
            for window in [21, 63]:

                feature_dict[f"{ticker}_skew_{window}"] = (
                    self.rolling_skew(returns[ticker], window).shift(1)
                )

                feature_dict[f"{ticker}_kurt_{window}"] = (
                    self.rolling_kurtosis(returns[ticker], window).shift(1)
                )

        # ==========================================================
        # 2. Volatility Term Structure Layer
        # ==========================================================

        vol_windows = [5, 21, 63, 252]

        for ticker in close.columns:

            vols = {
                w: self.realized_volatility(returns[ticker], w)
                for w in vol_windows
            }

            for short_w in vol_windows:
                for long_w in vol_windows:
                    if short_w < long_w:

                        spread_name = (
                            f"{ticker}_vol_spread_{short_w}_{long_w}"
                        )

                        feature_dict[spread_name] = (
                            vols[short_w] - vols[long_w]
                        ).shift(1)

        # ==========================================================
        # 3. Cross-Asset Dependency Layer
        # ==========================================================

        dependency_pairs: List[Tuple[str, str]] = [
            ("SPY", "TLT"),
            ("SPY", "GLD"),
            ("SPY", "USO"),
            ("TLT", "GLD"),
        ]

        for a, b in dependency_pairs:

            feature_dict[f"corr_{a}_{b}_63"] = (
                returns[a]
                .rolling(63)
                .corr(returns[b])
                .shift(1)
            )

            tracking_error = (
                (returns[a] - returns[b])
                .rolling(63)
                .std()
                .shift(1)
            )

            feature_dict[f"tracking_error_{a}_{b}"] = tracking_error

        # ==========================================================
        # 4. Microstructure & Flow Proxies
        # ==========================================================

        for ticker in close.columns:

            feature_dict[f"{ticker}_price_autocorr"] = (
                self.rolling_autocorr(close[ticker], 21).shift(1)
            )

            feature_dict[f"{ticker}_volume_std"] = (
                volume[ticker]
                .pct_change()
                .rolling(21)
                .std()
                .shift(1)
            )

        features = pd.DataFrame(feature_dict)

        # Drop NaNs caused by rolling windows
        features = features.dropna()

        self.features = features

        return features

In [ ]:
# manifold
%%writefile manifold.py
"""
manifold.py

Dimensionality reduction and latent manifold extraction.

The objective is to compress a high-dimensional, non-linear feature stack into
a lower-dimensional latent representation while preserving market geometry.

This module supports:
- Robust scaling for outlier-resistant normalization
- t-SNE manifold embedding
- Deep autoencoder latent compression


"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.manifold import TSNE
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, TensorDataset


@dataclass
class ManifoldConfig:
    latent_dim: int = 3
    epochs: int = 100
    batch_size: int = 64
    learning_rate: float = 1e-3
    tsne_perplexity: int = 30


class AutoEncoder(nn.Module):

    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.LeakyReLU(),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.LeakyReLU(),
            nn.Linear(32, latent_dim),
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.LayerNorm(32),
            nn.LeakyReLU(),
            nn.Linear(32, 64),
            nn.LayerNorm(64),
            nn.LeakyReLU(),
            nn.Linear(64, input_dim),
        )

    def forward(self, x: torch.Tensor):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction, latent


class ManifoldLearningEngine:

    def __init__(self, config: ManifoldConfig):
        self.config = config
        self.scaler = RobustScaler()

    def scale_features(self, X: pd.DataFrame) -> np.ndarray:
        return self.scaler.fit_transform(X)

    def tsne_embedding(self, X: np.ndarray) -> np.ndarray:

        tsne = TSNE(
            n_components=self.config.latent_dim,
            perplexity=self.config.tsne_perplexity,
            random_state=42,
        )

        return tsne.fit_transform(X)

    def autoencoder_embedding(self, X: np.ndarray) -> np.ndarray:

        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        X_tensor = torch.tensor(X, dtype=torch.float32)

        dataset = TensorDataset(X_tensor)
        loader = DataLoader(
            dataset,
            batch_size=self.config.batch_size,
            shuffle=True,
        )

        model = AutoEncoder(
            input_dim=X.shape[1],
            latent_dim=self.config.latent_dim,
        ).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=self.config.learning_rate,
        )

        criterion = nn.MSELoss()

        model.train()

        for epoch in range(self.config.epochs):

            total_loss = 0.0

            for batch in loader:

                batch_x = batch[0].to(device)

                optimizer.zero_grad()

                reconstruction, _ = model(batch_x)

                loss = criterion(reconstruction, batch_x)

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

            if epoch % 10 == 0:
                print(f"Epoch {epoch} | Loss: {total_loss:.4f}")

        model.eval()

        with torch.no_grad():
            _, latent = model(X_tensor.to(device))

        return latent.cpu().numpy()

In [ ]:
#vcluster
%%writefile cluster_engine.py

"""
cluster_engine.py

Regime discovery engine for non-stationary latent state estimation.

Implements:
- Gaussian Mixture Models
- HDBSCAN
- Jump-Diffusion Hidden Markov approximation
"""

from __future__ import annotations

from dataclasses import dataclass

import hdbscan
import numpy as np
from hmmlearn.hmm import GaussianHMM
from sklearn.mixture import GaussianMixture


@dataclass
class ClusterConfig:
    n_states: int = 4
    random_state: int = 42


class RegimeClusterEngine:

    def __init__(self, config: ClusterConfig):
        self.config = config

    def fit_gmm(
        self,
        X: np.ndarray,
    ) -> tuple[GaussianMixture, np.ndarray]:

        gmm = GaussianMixture(
            n_components=self.config.n_states,
            covariance_type="full",
            random_state=self.config.random_state,
        )

        gmm.fit(X)

        probs = gmm.predict_proba(X)

        return gmm, probs

    def fit_hdbscan(
        self,
        X: np.ndarray,
    ) -> tuple[hdbscan.HDBSCAN, np.ndarray]:

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=50,
            min_samples=10,
        )

        labels = clusterer.fit_predict(X)

        # -1 is explicitly treated as high-entropy transition state
        return clusterer, labels

    def fit_jump_hmm(
        self,
        X: np.ndarray,
    ) -> tuple[GaussianHMM, np.ndarray]:

        """
        Approximate Jump-Diffusion HMM.

        We augment the transition matrix with jump probabilities,
        allowing direct transitions to extreme states.

        This reflects market reflexivity and abrupt
        path-dependent regime transitions.
        """

        hmm = GaussianHMM(
            n_components=self.config.n_states,
            covariance_type="full",
            n_iter=500,
            random_state=self.config.random_state,
        )

        hmm.fit(X)

        # Simulated jump augmentation
        transmat = hmm.transmat_

        jump_strength = 0.05

        transmat += jump_strength * np.ones_like(transmat)

        transmat = transmat / transmat.sum(axis=1, keepdims=True)

        hmm.transmat_ = transmat

        hidden_states = hmm.predict(X)

        return hmm, hidden_states

In [ ]:
# validator
%%writefile validator.py
"""
validator.py

Information-theoretic validation engine.

Validates whether discovered regimes contain predictive and structural meaning.
"""

from __future__ import annotations

from dataclasses import dataclass

import networkx as nx
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.mixture import GaussianMixture


@dataclass
class ValidationConfig:
    max_clusters: int = 8


class RegimeValidator:

    def __init__(self, config: ValidationConfig):
        self.config = config

    def bic_analysis(self, X: np.ndarray):

        bic_scores = []

        for k in range(2, self.config.max_clusters + 1):

            model = GaussianMixture(
                n_components=k,
                covariance_type="full",
                random_state=42,
            )

            model.fit(X)

            bic_scores.append(model.bic(X))

        return bic_scores

    def compute_mst_length(
        self,
        returns: pd.DataFrame,
    ) -> float:

        corr = returns.corr().fillna(0)

        distance = np.sqrt(2 * (1 - corr))

        mst = minimum_spanning_tree(distance.values)

        return mst.sum()

    def regime_mst_test(
        self,
        returns: pd.DataFrame,
        labels: np.ndarray,
    ):

        results = {}

        for regime in np.unique(labels):

            mask = labels == regime

            subset = returns.loc[mask]

            if len(subset) < 20:
                continue

            mst_length = self.compute_mst_length(subset)

            results[int(regime)] = mst_length

        return results

    def regime_coherence_test(
        self,
        X: np.ndarray,
        labels: np.ndarray,
        target_vol: np.ndarray,
    ):

        split_idx = int(len(X) * 0.8)

        X_train = X[:split_idx]
        X_test = X[split_idx:]

        y_train = target_vol[:split_idx]
        y_test = target_vol[split_idx:]

        clf = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
        )

        clf.fit(X_train, y_train)

        preds = clf.predict(X_test)

        mse = mean_squared_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        # Naive benchmark
        naive = pd.Series(y_test).rolling(21).mean().bfill()

        naive_mse = mean_squared_error(y_test, naive)

        warning = None

        if mse >= naive_mse:
            warning = (
                "WARNING: Regime labels failed to outperform "
                "naive volatility forecast benchmark."
            )

        return {
            "mse": mse,
            "r2": r2,
            "naive_mse": naive_mse,
            "warning": warning,
        }

In [ ]:
%%writefile portfolio.py
"""
portfolio.py

Adaptive portfolio construction engine.

Implements regime-conditional capital allocation.

The portfolio logic explicitly recognizes:
- Non-stationarity
- Structural breaks
- Path-dependent covariance dynamics
- Reflexive market feedback loops
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import minimize


@dataclass
class PortfolioConfig:
    leverage: float = 1.5


class AdaptivePortfolioEngine:

    def __init__(self, config: PortfolioConfig):
        self.config = config

    @staticmethod
    def risk_parity_weights(cov: np.ndarray):

        n = cov.shape[0]

        def objective(w):

            portfolio_var = w.T @ cov @ w

            marginal = cov @ w

            contribution = w * marginal / portfolio_var

            return np.sum((contribution - 1 / n) ** 2)

        constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]

        bounds = [(0, 1)] * n

        x0 = np.ones(n) / n

        result = minimize(
            objective,
            x0=x0,
            bounds=bounds,
            constraints=constraints,
        )

        return result.x

    @staticmethod
    def minimum_variance_weights(cov: np.ndarray):

        inv_cov = np.linalg.pinv(cov)

        ones = np.ones(len(cov))

        w = inv_cov @ ones / (ones.T @ inv_cov @ ones)

        return w

    def allocate(
        self,
        returns: pd.DataFrame,
        states: np.ndarray,
    ) -> pd.Series:

        portfolio_returns = []

        assets = ["SPY", "TLT", "GLD", "USO", "XLF", "VIXY"]

        aligned_returns = returns[assets].iloc[-len(states):]

        for i in range(252, len(aligned_returns)):

            state = states[i]

            hist_mask = states[:i] == state

            hist_returns = aligned_returns.iloc[:i][hist_mask]

            if len(hist_returns) < 30:
                portfolio_returns.append(0.0)
                continue

            cov = hist_returns.cov().values

            # ======================================================
            # Regime-Specific Allocation Logic
            # ======================================================

            if state == 0:
                weights = (
                    self.risk_parity_weights(cov)
                    * self.config.leverage
                )

            elif state == 1:
                weights = self.minimum_variance_weights(cov)

            elif state == 2:
                # Tail-risk regime:
                # allocate heavily to volatility/cash
                weights = np.array(
                    [0.05, 0.40, 0.20, 0.0, 0.0, 0.35]
                )

            else:
                # High entropy/noise regime
                # Quadratic cost penalization
                weights = np.array(
                    [0.10, 0.30, 0.20, 0.0, 0.0, 0.40]
                )

            day_ret = aligned_returns.iloc[i].values @ weights

            portfolio_returns.append(day_ret)

        return pd.Series(
            portfolio_returns,
            index=aligned_returns.index[252:],
        )

In [ ]:
# main
%%writefile main.py
# main.py
# Google Colab Compatible End-to-End Regime Detection System

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

from scipy.optimize import minimize
from scipy.stats import skew, kurtosis
from scipy.sparse.csgraph import minimum_spanning_tree

from hmmlearn.hmm import GaussianHMM

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import hdbscan
import yfinance as yf


# ============================================================
# CONFIG
# ============================================================

TICKERS = ["SPY", "TLT", "GLD", "USO", "XLF", "VIXY"]

START_DATE = "2008-01-01"

LATENT_DIM = 3

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {DEVICE}")


# ============================================================
# DATA INGESTION
# ============================================================

print("\nDownloading market data...")

data = yf.download(
    TICKERS,
    start=START_DATE,
    auto_adjust=True,
    progress=False,
)

close = data["Close"]
volume = data["Volume"]

returns = np.log(close / close.shift(1)).dropna()

print("Data shape:", close.shape)


# ============================================================
# FEATURE ENGINEERING
# ============================================================

print("\nBuilding feature stack...")

features = {}

# ------------------------------------------------------------
# 1. Statistical Moments
# ------------------------------------------------------------

for ticker in TICKERS:

    for window in [21, 63]:

        features[f"{ticker}_skew_{window}"] = (
            returns[ticker]
            .rolling(window)
            .apply(lambda x: skew(x), raw=False)
            .shift(1)
        )

        features[f"{ticker}_kurt_{window}"] = (
            returns[ticker]
            .rolling(window)
            .apply(lambda x: kurtosis(x), raw=False)
            .shift(1)
        )

# ------------------------------------------------------------
# 2. Volatility Term Structure
# ------------------------------------------------------------

vol_windows = [5, 21, 63, 252]

for ticker in TICKERS:

    vols = {}

    for w in vol_windows:
        vols[w] = (
            returns[ticker]
            .rolling(w)
            .std() * np.sqrt(252)
        )

    for short_w in vol_windows:
        for long_w in vol_windows:

            if short_w < long_w:

                features[
                    f"{ticker}_vol_spread_{short_w}_{long_w}"
                ] = (
                    vols[short_w] - vols[long_w]
                ).shift(1)

# ------------------------------------------------------------
# 3. Cross Asset Dependency
# ------------------------------------------------------------

pairs = [
    ("SPY", "TLT"),
    ("SPY", "GLD"),
    ("SPY", "USO"),
    ("TLT", "GLD"),
]

for a, b in pairs:

    features[f"corr_{a}_{b}"] = (
        returns[a]
        .rolling(63)
        .corr(returns[b])
        .shift(1)
    )

    features[f"tracking_error_{a}_{b}"] = (
        (returns[a] - returns[b])
        .rolling(63)
        .std()
        .shift(1)
    )

# ------------------------------------------------------------
# 4. Flow/Microstructure Proxies
# ------------------------------------------------------------

for ticker in TICKERS:

    features[f"{ticker}_price_autocorr"] = (
        close[ticker]
        .rolling(21)
        .apply(lambda x: pd.Series(x).autocorr(), raw=False)
        .shift(1)
    )

    features[f"{ticker}_volume_std"] = (
        volume[ticker]
        .pct_change()
        .rolling(21)
        .std()
        .shift(1)
    )

features = pd.DataFrame(features).dropna()

print("Feature matrix shape:", features.shape)


# ============================================================
# ROBUST SCALING
# ============================================================

print("\nScaling features...")

scaler = RobustScaler()

X_scaled = scaler.fit_transform(features)


# ============================================================
# AUTOENCODER
# ============================================================

print("\nTraining autoencoder...")


class AutoEncoder(nn.Module):

    def __init__(self, input_dim, latent_dim):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.LeakyReLU(),

            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.LeakyReLU(),

            nn.Linear(32, latent_dim),
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.LayerNorm(32),
            nn.LeakyReLU(),

            nn.Linear(32, 64),
            nn.LayerNorm(64),
            nn.LeakyReLU(),

            nn.Linear(64, input_dim),
        )

    def forward(self, x):

        latent = self.encoder(x)

        reconstruction = self.decoder(latent)

        return reconstruction, latent


X_tensor = torch.tensor(
    X_scaled,
    dtype=torch.float32
)

dataset = TensorDataset(X_tensor)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
)

model = AutoEncoder(
    input_dim=X_scaled.shape[1],
    latent_dim=LATENT_DIM,
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

criterion = nn.MSELoss()

EPOCHS = 100

model.train()

for epoch in range(EPOCHS):

    total_loss = 0

    for batch in loader:

        batch_x = batch[0].to(DEVICE)

        optimizer.zero_grad()

        reconstruction, _ = model(batch_x)

        loss = criterion(reconstruction, batch_x)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss {total_loss:.4f}")

model.eval()

with torch.no_grad():

    _, latent = model(
        X_tensor.to(DEVICE)
    )

latent = latent.cpu().numpy()

print("Latent shape:", latent.shape)


# ============================================================
# GMM CLUSTERING
# ============================================================

print("\nFitting Gaussian Mixture Model...")

gmm = GaussianMixture(
    n_components=4,
    covariance_type="full",
    random_state=42,
)

gmm.fit(latent)

gmm_probs = gmm.predict_proba(latent)

gmm_states = gmm.predict(latent)


# ============================================================
# HDBSCAN
# ============================================================

print("\nRunning HDBSCAN...")

hdb = hdbscan.HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
)

hdb_labels = hdb.fit_predict(latent)

print(
    "Unique HDBSCAN labels:",
    np.unique(hdb_labels)
)


# ============================================================
# HMM REGIME MODEL
# ============================================================

print("\nFitting Jump-Diffusion HMM...")

hmm = GaussianHMM(
    n_components=4,
    covariance_type="full",
    n_iter=500,
    random_state=42,
)

hmm.fit(latent)

# Simulated jump augmentation

transmat = hmm.transmat_

transmat += 0.05 * np.ones_like(transmat)

transmat = (
    transmat
    / transmat.sum(axis=1, keepdims=True)
)

hmm.transmat_ = transmat

hidden_states = hmm.predict(latent)

print(
    "Hidden states:",
    np.unique(hidden_states)
)


# ============================================================
# BIC ANALYSIS
# ============================================================

print("\nBIC Analysis")

bic_scores = []

for k in range(2, 9):

    bic_model = GaussianMixture(
        n_components=k,
        covariance_type="full",
        random_state=42,
    )

    bic_model.fit(latent)

    bic = bic_model.bic(latent)

    bic_scores.append(bic)

    print(f"K={k} | BIC={bic:.2f}")


# ============================================================
# MST VALIDATION
# ============================================================

print("\nMST Regime Validation")

aligned_returns = returns.loc[features.index]

for regime in np.unique(hidden_states):

    mask = hidden_states == regime

    subset = aligned_returns.iloc[mask]

    if len(subset) < 20:
        continue

    corr = subset.corr().fillna(0)

    distance = np.sqrt(2 * (1 - corr))

    mst = minimum_spanning_tree(distance.values)

    print(
        f"Regime {regime} MST Length:",
        mst.sum()
    )


# ============================================================
# REGIME COHERENCE TEST
# ============================================================


print("\nRunning coherence test...")

# ------------------------------------------------------------
# Future realized volatility target
# ------------------------------------------------------------

target_vol = (
    aligned_returns["SPY"]
    .rolling(21)
    .std()
    .shift(-1)
)

# ------------------------------------------------------------
# Remove NaNs jointly from latent features and target
# ------------------------------------------------------------

valid_mask = ~target_vol.isna()

latent_clean = latent[valid_mask]

target_vol_clean = target_vol[valid_mask]

# ------------------------------------------------------------
# Train/Test Split
# ------------------------------------------------------------

split = int(len(latent_clean) * 0.8)

X_train = latent_clean[:split]
X_test = latent_clean[split:]

y_train = target_vol_clean.iloc[:split]
y_test = target_vol_clean.iloc[split:]

# ------------------------------------------------------------
# Random Forest Regime Coherence Model
# ------------------------------------------------------------

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
)

rf.fit(X_train, y_train)

preds = rf.predict(X_test)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

mse = mean_squared_error(y_test, preds)

r2 = r2_score(y_test, preds)

# ------------------------------------------------------------
# Naive Rolling Volatility Benchmark
# ------------------------------------------------------------

naive = (
    y_test
    .rolling(21)
    .mean()
    .bfill()
)

naive_mse = mean_squared_error(y_test, naive)

# ------------------------------------------------------------
# Print Results
# ------------------------------------------------------------

print(f"MSE: {mse:.6f}")

print(f"R²: {r2:.4f}")

print(f"Naive MSE: {naive_mse:.6f}")

if mse >= naive_mse:

    print(
        "\nWARNING: Regime model failed to "
        "outperform naive benchmark."
    )


# ============================================================
# PORTFOLIO ENGINE
# ============================================================

print("\nConstructing adaptive portfolio...")


def risk_parity_weights(cov):

    n = cov.shape[0]

    def objective(w):

        portfolio_var = w.T @ cov @ w

        marginal = cov @ w

        contribution = (
            w * marginal / portfolio_var
        )

        return np.sum(
            (contribution - 1 / n) ** 2
        )

    constraints = [
        {
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1,
        }
    ]

    bounds = [(0, 1)] * n

    x0 = np.ones(n) / n

    result = minimize(
        objective,
        x0,
        bounds=bounds,
        constraints=constraints,
    )

    return result.x


def minimum_variance_weights(cov):

    inv_cov = np.linalg.pinv(cov)

    ones = np.ones(len(cov))

    return (
        inv_cov @ ones
        / (ones.T @ inv_cov @ ones)
    )


portfolio_returns = []

assets = TICKERS

aligned = aligned_returns[assets]

for i in range(252, len(aligned)):

    state = hidden_states[i]

    hist_mask = hidden_states[:i] == state

    hist_returns = aligned.iloc[:i][hist_mask]

    if len(hist_returns) < 30:

        portfolio_returns.append(0)

        continue

    cov = hist_returns.cov().values

    # --------------------------------------------------------
    # Regime-conditioned allocations
    # --------------------------------------------------------

    if state == 0:

        weights = (
            risk_parity_weights(cov)
            * 1.5
        )

    elif state == 1:

        weights = minimum_variance_weights(cov)

    elif state == 2:

        weights = np.array(
            [0.05, 0.40, 0.20, 0.0, 0.0, 0.35]
        )

    else:

        weights = np.array(
            [0.10, 0.30, 0.20, 0.0, 0.0, 0.40]
        )

    daily_return = (
        aligned.iloc[i].values @ weights
    )

    portfolio_returns.append(daily_return)

strategy_returns = pd.Series(
    portfolio_returns,
    index=aligned.index[252:]
)


# ============================================================
# BENCHMARK
# ============================================================

benchmark = (
    0.60 * aligned["SPY"]
    + 0.40 * aligned["TLT"]
)

benchmark = benchmark.loc[strategy_returns.index]


# ============================================================
# PERFORMANCE CURVES
# ============================================================

strategy_curve = (
    1 + strategy_returns
).cumprod()

benchmark_curve = (
    1 + benchmark
).cumprod()


# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(14, 7))

plt.plot(
    strategy_curve.index,
    strategy_curve,
    label="Adaptive Regime Portfolio",
    linewidth=2,
)

plt.plot(
    benchmark_curve.index,
    benchmark_curve,
    label="60/40 Benchmark",
    linewidth=2,
)

plt.title(
    "Adaptive Regime Portfolio vs Benchmark"
)

plt.xlabel("Date")

plt.ylabel("Cumulative Return")

plt.grid(True)

plt.legend()

plt.show()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n==============================")
print("PIPELINE COMPLETE")
print("==============================")

print(
    f"Final Strategy Return: "
    f"{strategy_curve.iloc[-1]:.2f}x"
)

print(
    f"Final Benchmark Return: "
    f"{benchmark_curve.iloc[-1]:.2f}x"
)

In [ ]:
!python main.py